# Clasificacion binaria


Usa esta plantilla cuando la salida tenga dos clases.

Ejemplos: spam/no spam, enfermo/sano, fraude/no fraude, aprobado/suspenso.

Lo importante aquí es mirar la matriz de confusión. En binaria, los errores no valen siempre lo mismo.


In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TARGET = "target"  # CAMBIAR
POSITIVE_LABEL = 1  # CAMBIAR si tu clase positiva es "yes", "spam", "enfermo", etc.

df = pd.read_csv("data.csv")

print(df[TARGET].value_counts(dropna=False))

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)


## Modelo base robusto

`class_weight="balanced"` ayuda si una clase aparece mucho menos. Si las clases están equilibradas también suele funcionar bien, así que es una opción bastante segura en examen.


In [ ]:
numeric_cols = X_train.select_dtypes(include="number").columns
categorical_cols = X_train.select_dtypes(exclude="number").columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))]), categorical_cols),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)


In [ ]:
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(classification_report(y_test, y_pred, zero_division=0))

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, pos_label=POSITIVE_LABEL, zero_division=0)
recall = recall_score(y_test, y_pred, pos_label=POSITIVE_LABEL, zero_division=0)
f1 = f1_score(y_test, y_pred, pos_label=POSITIVE_LABEL, zero_division=0)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1:        {f1:.4f}")


## Cómo decidir qué métrica importa

- Medicina, seguridad, detectar positivos raros: prioriza `recall` porque no quieres perder positivos reales.
- Spam, bloqueo de cuentas, acusar fraude: prioriza `precision` porque no quieres falsos positivos.
- Sin contexto claro: usa `F1` porque equilibra precision y recall.
- `Accuracy` solo es cómoda si las clases están razonablemente balanceadas.


## Si falla

- `pos_label` da error: tu clase positiva no es `1`; cambia `POSITIVE_LABEL`.
- Accuracy alta pero recall bajo: el modelo ignora la clase minoritaria.
- Todo predice una clase: revisa desbalance, escalado y features.
- Error con strings: faltaba codificación de categóricas.
